# Redes Neuronales III – Práctica 2: Regularización (Semana 6)

**Caso de uso (sector salud):** clasificación de tumores de mama como benignos o malignos a partir de 30 características de núcleos celulares (dataset *Breast Cancer Wisconsin – Diagnostic*, 569 muestras). Uso exclusivamente académico: no sustituye un diagnóstico médico.

Temas: bias, variance, overfitting, L1/L2, Dropout, Data Augmentation, Early Stopping.

## 1. Datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42

# 1. Carga: Breast Cancer Wisconsin (Diagnostic), incluido en scikit-learn
data = load_breast_cancer(as_frame=True)
X = data.data
y = (1 - data.target).astype("int32")      # 1 = maligno, 0 = benigno

# 2. Distribucion estratificada 60 / 20 / 20 (train / dev / test)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
X_train, X_dev, y_train, y_dev = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=SEED)

# 3. Preprocesamiento: estandarizacion ajustada SOLO con train (evita fuga de datos)
scaler = StandardScaler().fit(X_train)
Xtr_s, Xdv_s, Xte_s = [scaler.transform(a).astype("float32")
                       for a in (X_train, X_dev, X_test)]
Xtr_raw = X_train.to_numpy(dtype="float32")     # sin escalar (solo para comparar)
Xdv_raw = X_dev.to_numpy(dtype="float32")
N_FEATURES = Xtr_s.shape[1]

resumen = pd.DataFrame({
    "conjunto": ["train", "dev", "test"],
    "muestras": [len(y_train), len(y_dev), len(y_test)],
    "malignos": [int(y_train.sum()), int(y_dev.sum()), int(y_test.sum())]})
resumen["% malignos"] = (100 * resumen["malignos"] / resumen["muestras"]).round(1)
print(resumen.to_string(index=False))
print("Caracteristicas:", N_FEATURES)

## 2. Modelo y utilidades

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

keras.utils.set_random_seed(SEED)

def build_model(units=(64, 32), use_bn=False, dropout=0.0, l1=0.0, l2=0.0, noise=0.0):
    """MLP binario. noise>0 agrega ruido gaussiano (data augmentation tabular)."""
    reg = regularizers.L1L2(l1=l1, l2=l2) if (l1 or l2) else None
    m = keras.Sequential([layers.Input(shape=(N_FEATURES,))])
    if noise:
        m.add(layers.GaussianNoise(noise))
    for u in units:
        m.add(layers.Dense(u, use_bias=not use_bn, kernel_regularizer=reg))
        if use_bn:
            m.add(layers.BatchNormalization())
        m.add(layers.Activation("relu"))
        if dropout:
            m.add(layers.Dropout(dropout))
    m.add(layers.Dense(1, activation="sigmoid"))
    return m

def compile_model(m, optimizer):
    m.compile(optimizer=optimizer, loss="binary_crossentropy",
              metrics=["accuracy",
                       keras.metrics.Recall(name="recall"),
                       keras.metrics.AUC(name="auc")])
    return m

def entrenar(make_model, make_opt, Xtr=None, Xdv=None, epochs=100, batch_size=32,
             callbacks=None, seeds=(1, 2, 3)):
    """Entrena con varias semillas y promedia, porque el dataset es pequeno."""
    Xtr = Xtr_s if Xtr is None else Xtr
    Xdv = Xdv_s if Xdv is None else Xdv
    filas, hist = [], None
    for s in seeds:
        keras.backend.clear_session()
        keras.utils.set_random_seed(s)
        m = compile_model(make_model(), make_opt())
        h = m.fit(Xtr, y_train, validation_data=(Xdv, y_dev), epochs=epochs,
                  batch_size=batch_size, callbacks=callbacks() if callbacks else [],
                  verbose=0)
        tr = m.evaluate(Xtr, y_train, verbose=0, return_dict=True)
        dv = m.evaluate(Xdv, y_dev, verbose=0, return_dict=True)
        filas.append([tr["accuracy"], dv["accuracy"], dv["recall"], dv["auc"],
                      len(h.history["loss"])])
        hist = h.history
    r = np.array(filas)
    resumen = {"acc_train": r[:, 0].mean(), "acc_dev": r[:, 1].mean(),
               "std_dev": r[:, 1].std(), "recall_dev": r[:, 2].mean(),
               "auc_dev": r[:, 3].mean(), "epocas": r[:, 4].mean()}
    return resumen, hist

## 3. Experimentos de regularización

In [ ]:
# Modelo deliberadamente grande para provocar overfitting en un dataset pequeno
GRANDE = (256, 128, 64)
adam = lambda: keras.optimizers.Adam(1e-3)
early = lambda: [keras.callbacks.EarlyStopping(monitor="val_loss", patience=15,
                                               restore_best_weights=True)]

experimentos = {
    "Baseline (sin regularizar)": (dict(units=GRANDE), None),
    "L2 (1e-3)":                  (dict(units=GRANDE, l2=1e-3), None),
    "L1 (1e-4)":                  (dict(units=GRANDE, l1=1e-4), None),
    "Dropout (0.3)":              (dict(units=GRANDE, dropout=0.3), None),
    "Data augmentation (ruido)":  (dict(units=GRANDE, noise=0.1), None),
    "Early stopping":             (dict(units=GRANDE), early),
    "L2 + Dropout + Early stop.": (dict(units=GRANDE, l2=1e-3, dropout=0.3), early),
}

tabla, curvas = {}, {}
for nombre, (kw, cb) in experimentos.items():
    tabla[nombre], curvas[nombre] = entrenar(
        lambda kw=kw: build_model(**kw), adam, epochs=150, callbacks=cb)
    t = tabla[nombre]
    print(f"{nombre:28s} train={t['acc_train']:.4f} dev={t['acc_dev']:.4f} "
          f"brecha={t['acc_train']-t['acc_dev']:.4f} epocas={t['epocas']:.0f}")

df = pd.DataFrame(tabla).T
df["brecha"] = df["acc_train"] - df["acc_dev"]
print(df.round(4))

## 4. Curvas de pérdida

In [ ]:
# Curvas de perdida: baseline vs. combinacion regularizada
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for a, nombre in zip(ax, ["Baseline (sin regularizar)", "L2 + Dropout + Early stop."]):
    a.plot(curvas[nombre]["loss"], label="train")
    a.plot(curvas[nombre]["val_loss"], label="dev")
    a.set_title(nombre); a.set_xlabel("Epoca"); a.legend(); a.grid(alpha=.3)
ax[0].set_ylabel("Perdida (binary cross-entropy)")
plt.tight_layout(); plt.show()

## 5. Diagnóstico bias / variance

*Compara el error en train y en dev de cada fila y clasifica: alto bias, alta variance, ambos o buen ajuste.*